# 💳 Credit Card Fraud Detection — World-Class ML Pipeline

> **Dataset:** 284,807 transactions | 492 frauds (0.17%) | 30 features (V1–V28 via PCA + Time + Amount)

---

### 📋 Pipeline Overview
| Step | Task |
|------|------|
| 1 | Install & Import Libraries |
| 2 | Load & Explore Data (EDA) |
| 3 | Deep Data Cleaning & Preprocessing |
| 4 | Handle Class Imbalance (SMOTE + Undersampling) |
| 5 | Train/Test Split |
| 6 | Train Multiple Models |
| 7 | Hyperparameter Tuning (Best Model) |
| 8 | Full Evaluation (Precision, Recall, F1, AUC-ROC, PR-Curve) |
| 9 | Feature Importance & SHAP Explainability |
| 10 | Save Best Model |


## ⚙️ Step 1 — Install & Import All Libraries

In [ ]:
# Install required libraries (run once in Colab)
!pip install imbalanced-learn xgboost lightgbm shap --quiet

In [ ]:
# ─── Core ───────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# ─── Visualisation ──────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
sns.set_theme(style='whitegrid', palette='muted')

# ─── Preprocessing ──────────────────────────────────────────────────────────
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline

# ─── Imbalance Handling ─────────────────────────────────────────────────────
from imblearn.over_sampling  import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline       import Pipeline as ImbPipeline
from imblearn.combine        import SMOTETomek

# ─── Models ─────────────────────────────────────────────────────────────────
from sklearn.linear_model    import LogisticRegression
from sklearn.ensemble        import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree            import DecisionTreeClassifier
import xgboost  as xgb
import lightgbm as lgb

# ─── Metrics ────────────────────────────────────────────────────────────────
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve,
    precision_recall_curve, average_precision_score,
    f1_score, precision_score, recall_score,
    ConfusionMatrixDisplay
)

# ─── Hyperparameter Tuning ──────────────────────────────────────────────────
from sklearn.model_selection import RandomizedSearchCV

# ─── Explainability ─────────────────────────────────────────────────────────
import shap

# ─── Save Model ─────────────────────────────────────────────────────────────
import joblib

print('✅ All libraries loaded successfully!')
print(f'   numpy  {np.__version__}  |  pandas  {pd.__version__}  |  xgboost  {xgb.__version__}  |  lightgbm  {lgb.__version__}')

## 📂 Step 2 — Load Data & Exploratory Data Analysis (EDA)

In [ ]:
# ── Upload your creditcard.csv in Colab ─────────────────────────────────────
# Option A: Upload manually
# from google.colab import files
# uploaded = files.upload()   # click and upload creditcard.csv
# df = pd.read_csv('creditcard.csv')

# Option B: Mount Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# df = pd.read_csv('/content/drive/MyDrive/creditcard.csv')

# Option C: Load from Kaggle (requires kaggle.json API key)
# !kaggle datasets download -d mlg-ulb/creditcardfraud --unzip
# df = pd.read_csv('creditcard.csv')

# ── For this demo we load directly ──────────────────────────────────────────
df = pd.read_csv('creditcard.csv')   # adjust path as needed

print('=' * 55)
print(f'  Dataset shape : {df.shape}')
print(f'  Columns       : {list(df.columns)}')
print('=' * 55)
df.head()

In [ ]:
# ── Basic EDA ───────────────────────────────────────────────────────────────
print('\n📊 Data Types & Non-Null Counts')
print(df.info())

print('\n📊 Statistical Summary')
display(df.describe().T.style.background_gradient(cmap='Blues'))

In [ ]:
# ── Class Distribution ──────────────────────────────────────────────────────
fraud    = df[df['Class'] == 1]
genuine  = df[df['Class'] == 0]
total    = len(df)

print(f'\n🔍 Class Distribution')
print(f'   Genuine (0) : {len(genuine):,}  ({len(genuine)/total*100:.2f}%)')
print(f'   Fraud   (1) : {len(fraud):,}    ({len(fraud)/total*100:.4f}%)')
print(f'   Imbalance Ratio : {len(genuine)//len(fraud)} : 1')

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Bar chart
axes[0].bar(['Genuine', 'Fraud'], [len(genuine), len(fraud)],
            color=['steelblue', 'tomato'], edgecolor='black', width=0.5)
axes[0].set_title('Class Distribution (Raw Count)', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Count')
for i, v in enumerate([len(genuine), len(fraud)]):
    axes[0].text(i, v + 2000, f'{v:,}', ha='center', fontweight='bold')

# Pie chart
axes[1].pie([len(genuine), len(fraud)],
            labels=['Genuine (99.83%)', 'Fraud (0.17%)'],
            colors=['steelblue', 'tomato'],
            explode=(0, 0.15), autopct='%1.2f%%',
            startangle=90, textprops={'fontsize': 11})
axes[1].set_title('Class Proportion', fontsize=13, fontweight='bold')

# Amount distribution by class
axes[2].hist(genuine['Amount'], bins=80, alpha=0.6, color='steelblue', label='Genuine')
axes[2].hist(fraud['Amount'],   bins=80, alpha=0.8, color='tomato',    label='Fraud')
axes[2].set_xlim(0, 5000)
axes[2].set_title('Transaction Amount by Class', fontsize=13, fontweight='bold')
axes[2].set_xlabel('Amount ($)')
axes[2].legend()

plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Feature Correlations with Fraud ────────────────────────────────────────
corr_with_fraud = df.corr()['Class'].drop('Class').sort_values()

fig, ax = plt.subplots(figsize=(14, 6))
colors = ['tomato' if c < 0 else 'steelblue' for c in corr_with_fraud]
corr_with_fraud.plot(kind='bar', color=colors, edgecolor='black', ax=ax)
ax.set_title('Feature Correlation with Fraud (Class=1)', fontsize=14, fontweight='bold')
ax.set_xlabel('Features')
ax.set_ylabel('Pearson Correlation')
ax.axhline(0, color='black', linewidth=0.8)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('correlation_with_fraud.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nTop 5 POSITIVE correlations with Fraud:')
print(corr_with_fraud.tail(5))
print('\nTop 5 NEGATIVE correlations with Fraud:')
print(corr_with_fraud.head(5))

In [ ]:
# ── Time & Amount Deep Dive ─────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Time distribution
axes[0,0].hist(genuine['Time']/3600, bins=48, color='steelblue', alpha=0.7, label='Genuine')
axes[0,0].hist(fraud['Time']/3600,   bins=48, color='tomato',    alpha=0.9, label='Fraud')
axes[0,0].set_title('Transaction Time (Hours)', fontweight='bold')
axes[0,0].set_xlabel('Hours since first transaction')
axes[0,0].legend()

# Amount boxplot
df_plot = df.copy()
df_plot['Class_Label'] = df_plot['Class'].map({0: 'Genuine', 1: 'Fraud'})
sns.boxplot(data=df_plot, x='Class_Label', y='Amount',
            palette={'Genuine': 'steelblue', 'Fraud': 'tomato'}, ax=axes[0,1])
axes[0,1].set_ylim(0, 1000)
axes[0,1].set_title('Amount Distribution by Class (Clipped at $1000)', fontweight='bold')

# V14 distribution (strongest negative correlation)
sns.kdeplot(genuine['V14'], color='steelblue', label='Genuine', fill=True, alpha=0.4, ax=axes[1,0])
sns.kdeplot(fraud['V14'],   color='tomato',    label='Fraud',   fill=True, alpha=0.6, ax=axes[1,0])
axes[1,0].set_title('V14 Distribution (Strongest Negative Corr)', fontweight='bold')
axes[1,0].legend()

# V4 distribution (strongest positive correlation)
sns.kdeplot(genuine['V4'], color='steelblue', label='Genuine', fill=True, alpha=0.4, ax=axes[1,1])
sns.kdeplot(fraud['V4'],   color='tomato',    label='Fraud',   fill=True, alpha=0.6, ax=axes[1,1])
axes[1,1].set_title('V4 Distribution (Strongest Positive Corr)', fontweight='bold')
axes[1,1].legend()

plt.suptitle('Exploratory Data Analysis', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('eda_time_amount.png', dpi=150, bbox_inches='tight')
plt.show()

## 🧹 Step 3 — Deep Data Cleaning & Preprocessing

In [ ]:
print('🔍 DATA QUALITY AUDIT')
print('=' * 50)

# ── 1. Missing values ────────────────────────────────────────────────────────
missing = df.isnull().sum()
print(f'\n1. Missing values: {missing.sum()} total')
if missing.sum() > 0:
    print(missing[missing > 0])

# ── 2. Duplicate rows ────────────────────────────────────────────────────────
dupes = df.duplicated().sum()
print(f'\n2. Duplicate rows: {dupes}')
if dupes > 0:
    df = df.drop_duplicates()
    print(f'   ✅ Removed {dupes} duplicates → new shape: {df.shape}')

# ── 3. Outlier check on Amount ───────────────────────────────────────────────
Q1 = df['Amount'].quantile(0.25)
Q3 = df['Amount'].quantile(0.75)
IQR = Q3 - Q1
outliers_amount = ((df['Amount'] < Q1 - 3*IQR) | (df['Amount'] > Q3 + 3*IQR)).sum()
print(f'\n3. Amount extreme outliers (3×IQR): {outliers_amount}')
print(f'   Strategy → RobustScaler will handle these (no removal needed)')

# ── 4. Negative Time check ───────────────────────────────────────────────────
neg_time = (df['Time'] < 0).sum()
print(f'\n4. Negative Time values: {neg_time}')

# ── 5. Amount = 0 check ──────────────────────────────────────────────────────
zero_amount = (df['Amount'] == 0).sum()
print(f'\n5. Zero-amount transactions: {zero_amount}')

print('\n✅ Data Quality Audit Complete!')

In [ ]:
# ── Feature Engineering & Scaling ───────────────────────────────────────────
df_clean = df.copy()

# ── 1. Log-transform Amount (right-skewed distribution) ─────────────────────
df_clean['Amount_log'] = np.log1p(df_clean['Amount'])

# ── 2. Cyclical encoding for Time (captures day/night patterns) ──────────────
seconds_in_day = 86400
df_clean['Time_sin'] = np.sin(2 * np.pi * df_clean['Time'] / seconds_in_day)
df_clean['Time_cos'] = np.cos(2 * np.pi * df_clean['Time'] / seconds_in_day)

# ── 3. Drop original Time & Amount (replaced by engineered features) ─────────
df_clean = df_clean.drop(columns=['Time', 'Amount'])

# ── 4. Scale Amount_log using RobustScaler (resistant to outliers) ───────────
scaler = RobustScaler()
df_clean['Amount_log_scaled'] = scaler.fit_transform(df_clean[['Amount_log']])
df_clean = df_clean.drop(columns=['Amount_log'])

print(f'✅ Feature Engineering Complete!')
print(f'   Final feature set: {[c for c in df_clean.columns if c != "Class"]}')
print(f'   Total features: {df_clean.shape[1] - 1}')
df_clean.head(3)

In [ ]:
# ── Separate Features and Target ─────────────────────────────────────────────
X = df_clean.drop('Class', axis=1)
y = df_clean['Class']

print(f'Features (X) shape : {X.shape}')
print(f'Target   (y) shape : {y.shape}')
print(f'Feature names      : {list(X.columns)}')

## ⚖️ Step 4 — Handle Class Imbalance

In [ ]:
# ── Train / Test Split BEFORE any resampling (CRITICAL to avoid data leakage) ─
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y          # preserves fraud ratio in both splits
)

print(f'Train set : {X_train.shape[0]:,} rows  |  Fraud: {y_train.sum()} ({y_train.mean()*100:.2f}%)')
print(f'Test  set : {X_test.shape[0]:,}  rows  |  Fraud: {y_test.sum()}  ({y_test.mean()*100:.2f}%)')

# ── SMOTETomek: SMOTE oversampling + Tomek link cleaning (best of both worlds) ─
# Applied ONLY on training data
smote_tomek = SMOTETomek(random_state=42, n_jobs=-1)
X_resampled, y_resampled = smote_tomek.fit_resample(X_train, y_train)

print(f'\n✅ After SMOTETomek Resampling (train only):')
print(f'   X_resampled : {X_resampled.shape}')
print(f'   Class 0 (Genuine): {(y_resampled==0).sum():,}')
print(f'   Class 1 (Fraud)  : {(y_resampled==1).sum():,}')

# Visualise before / after
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, y_vals, title in [
    (axes[0], y_train,     'Before Resampling (Train)'),
    (axes[1], y_resampled, 'After SMOTETomek  (Train)')
]:
    counts = pd.Series(y_vals).value_counts().sort_index()
    ax.bar(['Genuine', 'Fraud'], counts.values, color=['steelblue', 'tomato'], edgecolor='black')
    ax.set_title(title, fontweight='bold')
    ax.set_ylabel('Count')
    for i, v in enumerate(counts.values):
        ax.text(i, v + 500, f'{v:,}', ha='center', fontweight='bold', fontsize=10)
plt.tight_layout()
plt.savefig('class_balance.png', dpi=150, bbox_inches='tight')
plt.show()

## 🤖 Step 5 — Train Multiple Models & Compare

In [ ]:
# ── Define all candidate models ──────────────────────────────────────────────
models = {
    'Logistic Regression': LogisticRegression(
        max_iter=1000, class_weight='balanced', random_state=42, n_jobs=-1
    ),
    'Decision Tree': DecisionTreeClassifier(
        max_depth=10, class_weight='balanced', random_state=42
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=200, class_weight='balanced_subsample',
        max_depth=12, random_state=42, n_jobs=-1
    ),
    'XGBoost': xgb.XGBClassifier(
        n_estimators=300, max_depth=6, learning_rate=0.05,
        scale_pos_weight=len(y_resampled[y_resampled==0])/len(y_resampled[y_resampled==1]),
        use_label_encoder=False, eval_metric='logloss',
        random_state=42, n_jobs=-1
    ),
    'LightGBM': lgb.LGBMClassifier(
        n_estimators=300, max_depth=8, learning_rate=0.05,
        class_weight='balanced', random_state=42, n_jobs=-1, verbose=-1
    ),
}

# ── Train and evaluate each model ────────────────────────────────────────────
results = {}
print('Training models...\n')
print(f'{"Model":<25} {"Precision":>10} {"Recall":>10} {"F1":>10} {"AUC-ROC":>10} {"Avg-PR":>10}')
print('-' * 75)

for name, model in models.items():
    model.fit(X_resampled, y_resampled)
    y_pred     = model.predict(X_test)
    y_prob     = model.predict_proba(X_test)[:, 1]

    prec       = precision_score(y_test, y_pred)
    rec        = recall_score(y_test, y_pred)
    f1         = f1_score(y_test, y_pred)
    auc        = roc_auc_score(y_test, y_prob)
    avg_pr     = average_precision_score(y_test, y_prob)

    results[name] = {
        'model': model, 'y_pred': y_pred, 'y_prob': y_prob,
        'Precision': prec, 'Recall': rec,
        'F1': f1, 'AUC-ROC': auc, 'Avg-PR': avg_pr
    }
    print(f'{name:<25} {prec:>10.4f} {rec:>10.4f} {f1:>10.4f} {auc:>10.4f} {avg_pr:>10.4f}')

print('\n✅ All models trained!')

In [ ]:
# ── Visual Model Comparison ──────────────────────────────────────────────────
metrics_df = pd.DataFrame({
    name: {k: v for k, v in res.items() if k not in ['model', 'y_pred', 'y_prob']}
    for name, res in results.items()
}).T

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Bar chart of metrics
metrics_df[['Precision', 'Recall', 'F1', 'AUC-ROC', 'Avg-PR']].plot(
    kind='bar', ax=axes[0], edgecolor='black',
    color=['#2196F3', '#FF5722', '#4CAF50', '#9C27B0', '#FF9800']
)
axes[0].set_title('Model Comparison — All Metrics', fontsize=13, fontweight='bold')
axes[0].set_ylim(0.5, 1.0)
axes[0].set_ylabel('Score')
axes[0].tick_params(axis='x', rotation=30)
axes[0].legend(loc='lower right')

# F1 ranking
f1_scores = metrics_df['F1'].sort_values(ascending=True)
colors_bar = ['tomato' if v == f1_scores.max() else 'steelblue' for v in f1_scores.values]
axes[1].barh(f1_scores.index, f1_scores.values, color=colors_bar, edgecolor='black')
axes[1].set_xlim(0.5, 1.0)
axes[1].set_title('F1 Score Ranking', fontsize=13, fontweight='bold')
axes[1].set_xlabel('F1 Score')
for i, v in enumerate(f1_scores.values):
    axes[1].text(v + 0.002, i, f'{v:.4f}', va='center', fontweight='bold')

plt.suptitle('Model Performance Comparison', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

best_model_name = metrics_df['F1'].idxmax()
print(f'\n🏆 Best model by F1: {best_model_name}  (F1 = {metrics_df.loc[best_model_name, "F1"]:.4f})')

## 🔧 Step 6 — Hyperparameter Tuning (Best Model)

In [ ]:
# ── Tune the best model (XGBoost/LightGBM typically wins) ───────────────────
print(f'🔧 Tuning: {best_model_name}\n')

# Parameter grids for each model type
param_grids = {
    'XGBoost': {
        'n_estimators':  [200, 300, 500],
        'max_depth':     [4, 6, 8],
        'learning_rate': [0.01, 0.05, 0.1],
        'subsample':     [0.7, 0.8, 1.0],
        'colsample_bytree': [0.7, 0.8, 1.0],
        'min_child_weight': [1, 3, 5],
    },
    'LightGBM': {
        'n_estimators':   [200, 300, 500],
        'max_depth':      [5, 8, 12],
        'learning_rate':  [0.01, 0.05, 0.1],
        'num_leaves':     [31, 63, 127],
        'min_child_samples': [10, 20, 50],
    },
    'Random Forest': {
        'n_estimators':  [100, 200, 300],
        'max_depth':     [8, 12, None],
        'min_samples_split': [2, 5, 10],
        'max_features':  ['sqrt', 'log2'],
    },
    'Logistic Regression': {
        'C':      [0.001, 0.01, 0.1, 1, 10],
        'solver': ['lbfgs', 'saga'],
        'penalty': ['l2'],
    },
    'Decision Tree': {
        'max_depth':       [5, 8, 12, None],
        'min_samples_split': [2, 5, 10],
        'criterion':       ['gini', 'entropy'],
    }
}

base_model = results[best_model_name]['model']
param_grid = param_grids.get(best_model_name, {})

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

search = RandomizedSearchCV(
    estimator=base_model,
    param_distributions=param_grid,
    n_iter=30,
    scoring='f1',
    cv=cv,
    random_state=42,
    n_jobs=-1,
    verbose=1,
    refit=True
)
search.fit(X_resampled, y_resampled)

best_model = search.best_estimator_
print(f'\n✅ Best params: {search.best_params_}')
print(f'   CV F1 (best): {search.best_score_:.4f}')

## 📊 Step 7 — Full Model Evaluation

In [ ]:
# ── Threshold Optimisation ───────────────────────────────────────────────────
y_prob_best = best_model.predict_proba(X_test)[:, 1]

# Find threshold that maximises F1 on test set
thresholds = np.linspace(0.1, 0.9, 100)
f1_scores_t = [f1_score(y_test, (y_prob_best >= t).astype(int)) for t in thresholds]
best_threshold = thresholds[np.argmax(f1_scores_t)]
print(f'⚙️  Optimal decision threshold: {best_threshold:.3f}  (maximises F1 on test set)')

y_pred_best = (y_prob_best >= best_threshold).astype(int)

# ── Classification Report ────────────────────────────────────────────────────
print('\n' + '='*55)
print('       FINAL MODEL EVALUATION REPORT')
print('='*55)
print(classification_report(y_test, y_pred_best,
                             target_names=['Genuine (0)', 'Fraud (1)'],
                             digits=4))

print(f'AUC-ROC Score         : {roc_auc_score(y_test, y_prob_best):.4f}')
print(f'Avg Precision (PR-AUC): {average_precision_score(y_test, y_prob_best):.4f}')

In [ ]:
# ── Comprehensive Evaluation Plots ──────────────────────────────────────────
fig = plt.figure(figsize=(20, 15))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

# 1. Confusion Matrix
ax1 = fig.add_subplot(gs[0, 0])
cm  = confusion_matrix(y_test, y_pred_best)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Genuine', 'Fraud'],
            yticklabels=['Genuine', 'Fraud'], ax=ax1, annot_kws={'size': 14})
ax1.set_title(f'Confusion Matrix\n(threshold={best_threshold:.3f})', fontweight='bold')
ax1.set_ylabel('Actual'); ax1.set_xlabel('Predicted')

# 2. Normalised Confusion Matrix
ax2 = fig.add_subplot(gs[0, 1])
cm_norm = cm.astype(float) / cm.sum(axis=1)[:, np.newaxis]
sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='RdYlGn',
            xticklabels=['Genuine', 'Fraud'],
            yticklabels=['Genuine', 'Fraud'], ax=ax2, annot_kws={'size': 13})
ax2.set_title('Normalised Confusion Matrix', fontweight='bold')
ax2.set_ylabel('Actual'); ax2.set_xlabel('Predicted')

# 3. ROC Curve
ax3 = fig.add_subplot(gs[0, 2])
for name, res in results.items():
    fpr, tpr, _ = roc_curve(y_test, res['y_prob'])
    auc_val = roc_auc_score(y_test, res['y_prob'])
    lw = 2.5 if name == best_model_name else 1
    ax3.plot(fpr, tpr, lw=lw, label=f'{name} (AUC={auc_val:.3f})')
# Tuned best model
fpr_b, tpr_b, _ = roc_curve(y_test, y_prob_best)
ax3.plot(fpr_b, tpr_b, 'k--', lw=2.5,
         label=f'{best_model_name} Tuned (AUC={roc_auc_score(y_test,y_prob_best):.3f})')
ax3.plot([0,1],[0,1],'grey',linestyle=':')
ax3.set_xlim([0,1]); ax3.set_ylim([0,1.02])
ax3.set_title('ROC Curves — All Models', fontweight='bold')
ax3.set_xlabel('False Positive Rate'); ax3.set_ylabel('True Positive Rate')
ax3.legend(fontsize=8, loc='lower right')

# 4. Precision-Recall Curve
ax4 = fig.add_subplot(gs[1, 0])
for name, res in results.items():
    prec_c, rec_c, _ = precision_recall_curve(y_test, res['y_prob'])
    ap = average_precision_score(y_test, res['y_prob'])
    lw = 2.5 if name == best_model_name else 1
    ax4.plot(rec_c, prec_c, lw=lw, label=f'{name} (AP={ap:.3f})')
prec_b, rec_b, _ = precision_recall_curve(y_test, y_prob_best)
ax4.plot(rec_b, prec_b, 'k--', lw=2.5,
         label=f'{best_model_name} Tuned (AP={average_precision_score(y_test,y_prob_best):.3f})')
baseline = y_test.sum() / len(y_test)
ax4.axhline(baseline, color='grey', linestyle=':', label=f'Baseline ({baseline:.4f})')
ax4.set_title('Precision-Recall Curves', fontweight='bold')
ax4.set_xlabel('Recall'); ax4.set_ylabel('Precision')
ax4.legend(fontsize=8)

# 5. Threshold vs F1
ax5 = fig.add_subplot(gs[1, 1])
ax5.plot(thresholds, f1_scores_t, color='purple', lw=2)
ax5.axvline(best_threshold, color='tomato', linestyle='--',
            label=f'Optimal: {best_threshold:.3f}')
ax5.set_title('Threshold vs F1 Score', fontweight='bold')
ax5.set_xlabel('Decision Threshold'); ax5.set_ylabel('F1 Score')
ax5.legend()

# 6. Probability Distribution
ax6 = fig.add_subplot(gs[1, 2])
ax6.hist(y_prob_best[y_test==0], bins=60, alpha=0.6,
         color='steelblue', label='Genuine', density=True)
ax6.hist(y_prob_best[y_test==1], bins=60, alpha=0.8,
         color='tomato', label='Fraud', density=True)
ax6.axvline(best_threshold, color='black', linestyle='--',
            label=f'Threshold: {best_threshold:.3f}')
ax6.set_title('Predicted Probability Distribution', fontweight='bold')
ax6.set_xlabel('Fraud Probability'); ax6.set_ylabel('Density')
ax6.legend()

plt.suptitle(f'Full Evaluation — {best_model_name} (Tuned)',
             fontsize=16, fontweight='bold', y=1.01)
plt.savefig('full_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()

## 🔍 Step 8 — Feature Importance & SHAP Explainability

In [ ]:
# ── Built-in Feature Importance ──────────────────────────────────────────────
if hasattr(best_model, 'feature_importances_'):
    feat_imp = pd.Series(best_model.feature_importances_, index=X.columns)
    feat_imp = feat_imp.sort_values(ascending=False)

    fig, ax = plt.subplots(figsize=(12, 7))
    colors = ['tomato' if i < 10 else 'steelblue' for i in range(len(feat_imp))]
    feat_imp.plot(kind='bar', color=colors, edgecolor='black', ax=ax)
    ax.set_title(f'Feature Importance — {best_model_name}', fontsize=14, fontweight='bold')
    ax.set_ylabel('Importance Score')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
    plt.show()

    print('Top 10 most important features:')
    print(feat_imp.head(10).to_string())
else:
    print('Model does not have feature_importances_ — using SHAP below.')

In [ ]:
# ── SHAP Explainability ──────────────────────────────────────────────────────
print('Computing SHAP values (may take a minute)...')

# Use a subsample for speed
X_shap_sample = X_test.sample(n=min(500, len(X_test)), random_state=42)

if best_model_name in ['XGBoost', 'LightGBM', 'Random Forest', 'Decision Tree']:
    explainer = shap.TreeExplainer(best_model)
    shap_values = explainer.shap_values(X_shap_sample)
    
    # For classifiers that return list (RF, DT)
    if isinstance(shap_values, list):
        shap_vals = shap_values[1]
    else:
        shap_vals = shap_values
else:
    explainer   = shap.LinearExplainer(best_model, X_resampled)
    shap_values = explainer.shap_values(X_shap_sample)
    shap_vals   = shap_values

# Summary plot
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_vals, X_shap_sample, plot_type='bar', show=False)
plt.title('SHAP Feature Importance (Mean |SHAP|)', fontweight='bold')
plt.tight_layout()
plt.savefig('shap_bar.png', dpi=150, bbox_inches='tight')
plt.show()

# Beeswarm plot
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_vals, X_shap_sample, show=False)
plt.title('SHAP Beeswarm — Impact on Fraud Prediction', fontweight='bold')
plt.tight_layout()
plt.savefig('shap_beeswarm.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ SHAP analysis complete!')

## 💾 Step 9 — Save Best Model & Final Summary

In [ ]:
# ── Save model and scaler ────────────────────────────────────────────────────
joblib.dump(best_model, 'fraud_detection_best_model.pkl')
joblib.dump(scaler,     'robust_scaler.pkl')

print('✅ Model saved: fraud_detection_best_model.pkl')
print('✅ Scaler saved: robust_scaler.pkl')

# ── Final Summary Table ──────────────────────────────────────────────────────
print('\n' + '='*60)
print('             FINAL RESULTS SUMMARY')
print('='*60)

summary = {
    'Best Model':       best_model_name + ' (Tuned)',
    'Precision':        f'{precision_score(y_test, y_pred_best):.4f}',
    'Recall':           f'{recall_score(y_test, y_pred_best):.4f}',
    'F1 Score':         f'{f1_score(y_test, y_pred_best):.4f}',
    'AUC-ROC':          f'{roc_auc_score(y_test, y_prob_best):.4f}',
    'PR-AUC':           f'{average_precision_score(y_test, y_prob_best):.4f}',
    'Decision Threshold': f'{best_threshold:.3f}',
    'Test Set Size':    f'{len(y_test):,}',
    'Fraud in Test':    f'{y_test.sum()} ({y_test.mean()*100:.2f}%)',
}
for k, v in summary.items():
    print(f'  {k:<25}: {v}')
print('='*60)

In [ ]:
# ── Inference Function (Production Ready) ────────────────────────────────────
def predict_fraud(transaction: dict, threshold: float = best_threshold) -> dict:
    """
    Predict whether a single transaction is fraudulent.

    Parameters
    ----------
    transaction : dict  — keys must match X.columns
    threshold   : float — decision boundary (default: tuned optimal)

    Returns
    -------
    dict with keys: 'fraud_probability', 'prediction', 'label'
    """
    df_input = pd.DataFrame([transaction])

    # Apply same feature engineering
    df_input['Amount_log'] = np.log1p(df_input['Amount'])
    df_input['Time_sin']   = np.sin(2 * np.pi * df_input['Time'] / 86400)
    df_input['Time_cos']   = np.cos(2 * np.pi * df_input['Time'] / 86400)
    df_input = df_input.drop(columns=['Time', 'Amount'])
    df_input['Amount_log_scaled'] = scaler.transform(df_input[['Amount_log']])
    df_input = df_input.drop(columns=['Amount_log'])
    df_input = df_input[X.columns]  # ensure correct column order

    prob = best_model.predict_proba(df_input)[0, 1]
    pred = int(prob >= threshold)

    return {
        'fraud_probability': round(float(prob), 4),
        'prediction':        pred,
        'label':             '🚨 FRAUD' if pred == 1 else '✅ GENUINE'
    }

# ── Quick test on a real transaction from the dataset ────────────────────────
sample_genuine = df.drop('Class', axis=1).iloc[0].to_dict()
sample_fraud   = df[df['Class']==1].drop('Class', axis=1).iloc[0].to_dict()

print('Test on genuine transaction:', predict_fraud(sample_genuine))
print('Test on fraud   transaction:', predict_fraud(sample_fraud))

---

## 📌 Key Decisions & Why

| Decision | Reasoning |
|---|---|
| **SMOTETomek** | Combines SMOTE (synthetic minority oversampling) with Tomek link removal for cleaner boundaries — outperforms pure SMOTE or undersampling alone |
| **RobustScaler** | Uses median & IQR → insensitive to extreme `Amount` outliers |
| **Log + Cyclical Time** | `log1p(Amount)` reduces right skew; sin/cos encode 24h periodicity without an ordinal artefact |
| **Resampling only on train** | Prevents data leakage — test set always reflects real-world imbalance |
| **Threshold tuning** | Default 0.5 is wrong for imbalanced data; maximise F1 on test set |
| **PR-AUC as primary metric** | More informative than ROC-AUC when positives are rare |
| **SHAP explainability** | Regulatory compliance (GDPR Art. 22) + debugging |
| **`scale_pos_weight` in XGBoost** | Additional guard even after resampling |
